# SOUL EXTER — autonomous trading council on a free Kaggle GPU

One notebook runs the whole product: the market feed, the fly-brain hunter, the six LLM
desks, the 3D trading floor AND the web UI, exposed through a public URL.

**Settings to use on Kaggle**
- Accelerator: *GPU T4 x2* (free tier) — the floor also runs on CPU, just slower.
- Internet: **On** (needed for the optional installs and the venue price feed).
- No API keys required: the six desks have a built-in analyst engine. Paste keys in the
  UI's *Settings → LLM Council* tab to promote any seat to a hosted open-source model.

**What you get**
- `/api/*` REST + `/ws` live frame stream (12 Hz) behind one port.
- The React/three.js floor served from the same origin.
- A public link of the form `https://<slug>-8000.ngrok-free.app` to open on any device.

## 1 · Get the code

In [ ]:
%cd /kaggle/working
import os
REPO = 'https://github.com/Naserkhan07/soul_exter.git'
if not os.path.isdir('soul_exter'):
    !git clone --depth 1 $REPO
%cd /kaggle/working/soul_exter
!git log --oneline -1 && ls

## 2 · Install what the floor needs
FastAPI + uvicorn serve the API and the built UI; the rest is already on Kaggle images.

In [ ]:
!pip install -q fastapi 'uvicorn[standard]' httpx pyngrok
!python -c "import fastapi, uvicorn, httpx; print('api deps ok')"
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## 3 · Build the interface (once per session)
Vite builds the React/three.js floor into `frontend/dist`, which the API server mounts,
so the browser talks to the same origin — no CORS, no second port.

In [ ]:
%cd /kaggle/working/soul_exter/frontend
!node --version
!npm install --silent
!npm run build
%cd /kaggle/working/soul_exter

## 4 · Prove the engine makes money before you trust it (optional, ~2 min)
This runs the whole pipeline headless and prints the realised book: accepted trades,
their P&L, and the counterfactual on the tickets the council vetoed.

In [ ]:
%cd /kaggle/working/soul_exter/backend
!python tests/track_record.py 90 30

## 5 · Start the floor (API + 3D interface on one port)

In [ ]:
import subprocess, time, os, sys
os.environ.setdefault('SOUL_EXTER_SETTINGS', '/kaggle/working/soul_exter_settings.json')
os.environ.setdefault('SOUL_EXTER_WEIGHTS', '/kaggle/working/soul_exter/backend/soul_exter_scorecard_weights.json')

server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'soul_exter.api.server:app',
     '--host', '0.0.0.0', '--port', '8000', '--log-level', 'info'],
    cwd='/kaggle/working/soul_exter/backend')
time.sleep(8)
print('server pid', server.pid, 'running:', server.poll() is None)
!curl -s localhost:8000/api/health

## 6 · Publish it
Grab a free authtoken at https://dashboard.ngrok.com/get-started/your-authtoken and paste
it below. The printed URL opens the full trading floor — desks, cabins, council, debate room.

In [ ]:
NGROK_TOKEN = ''      # <-- paste your token, or leave empty to use Kaggle's port proxy
if NGROK_TOKEN:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    tunnel = ngrok.connect(8000, 'http')
    print('OPEN THIS  ->', tunnel.public_url)
else:
    print('Open the Kaggle port proxy for port 8000 (Add-ons → Link to port 8000),'
          ' then append /docs or / to the URL.')

## 7 · Drive the floor from the notebook (optional)
Everything the UI does is a REST call — useful for scripting, sweeps and screenshots.

In [ ]:
import requests, json
B = 'http://localhost:8000'
print(json.dumps(requests.get(B + '/api/health').json(), indent=1))

# enable specific markets (per-pair checkboxes live in the UI; same call here)
pairs = ['EURUSD', 'GBPJPY', 'BTCUSD', 'ETHUSD', 'AAPL', 'NVDA', 'SPX', 'NDX', 'GC', 'CL']
print(requests.post(B + '/api/settings', json={'enabled_symbols': pairs}).json()['settings']['enabled_symbols'])

# speed the floor up, force a strike, read the book
requests.post(B + '/api/control', json={'action': 'speed', 'value': 8})
print(requests.post(B + '/api/control', json={'action': 'strike'}).json()['ok'])
print(json.dumps(requests.get(B + '/api/analytics').json()['stats'], indent=1))

## 8 · Optional: promote a desk to a hosted open-source model
Each of the six seats has a name, a specialty and an editable API key field in
*Settings → LLM Council*. The same can be done over REST:

In [ ]:
requests.post(B + '/api/seats/judge_trend', json={
    'provider': 'openrouter',            # or together / groq / deepseek / mistral / ollama
    'model': 'meta-llama/llama-3.1-70b-instruct',
    'api_key': 'sk-or-...',              # stored on this machine only
})
print(requests.post(B + '/api/seats/judge_trend/test').json())

## 9 · Keep it alive
Kaggle sessions time out; run the cell below in a loop if you want the floor up for hours.

In [ ]:
import time, requests
for i in range(24):
    time.sleep(300)
    try:
        st = requests.get(B + '/api/state', timeout=5).json()
        print(f"{i}: clock {st['clock']/60:.1f} min · tickets {len(st['trades'])} "
              f"· stats {st['stats']['accepted']} acc / {st['stats']['rejected']} veto "
              f"· pnl {st['stats']['pnl_r']:+.2f}R")
    except Exception as e:
        print('floor unreachable:', e)